In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "cuckfonst/affine-exciter-3"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
model = AutoModelForCausalLM(model_name)



In [ ]:
messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing well, thanks!"},
    {"role": "user", "content": "What's the weather?"},
    # {"role": "assistant", "content": "I don't have access to weather data."}
]


In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,  # Switches between thinking and non-thinking modes. Default is True.
    # return_tensor="pt"
)
text

In [ ]:
tokens = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt")

In [ ]:
tokens.shape[1]
# len(tokens)

In [ ]:
text1 = tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "Hello, how are you?"}
    ],
    tokenize=True,
    add_generation_prompt=False,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)

In [ ]:
text2 = tokenizer.apply_chat_template(
    [
        {"role": "assistant", "content": "I'm doing well, thanks!"}
    ],
    tokenize=True,
    add_generation_prompt=False,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)

In [ ]:
text1 + text2

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


In [ ]:

# prepare the model input
prompt = "what is 27+46"
messages = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": "I don't know"},
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": "I don't know"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)


In [ ]:

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


In [ ]:
text

In [ ]:
tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")